# Test Histogram Display

In [1]:
#include <TH1F.h>
#include <TCanvas.h>
#include <TRandom.h>

In [2]:
void createHistogram()
{
    TH1F *hist = new TH1F("Statistics Legend", "Histogram Title", 100, 0, 100);
    for (int i = 0; i < 10000; i++) {
        hist->Fill(gRandom->Gaus(50, 10));
    }
    
    TCanvas *test_canvas = new TCanvas("Test Canvas", "Canvas", 800, 600);
    hist->Draw();
    test_canvas->Draw();
}

In [3]:
createHistogram();

# Test File Read

In [4]:
#include <ROOT/RDataFrame.hxx>
#include <TFile.h>
#include <TTree.h>
#include <iostream>

In [5]:
void read_file(const std::string& filename)
{
    TFile *file = TFile::Open(filename.c_str());

    std::cout << "reading the file:" << filename.c_str() << std::endl;

    if (!file || file->IsZombie())
    {
        std::cerr << "Failed to open ROOT file" << std::endl;
        return;
    }

    std::cout << "successfully opened the file" << std::endl;

    TTree *tree = (TTree*)file->Get("tree");

    if (!tree)
    {
        std::cerr << "Tree not found!" << std::endl;
        return;
    }

    std::cout << "successfully opened the tree" << std::endl;

    std::cout << "Tree has " << tree->GetEntries() << " entries." << std::endl;

    std::cout << "-------------------------------------" << std::endl;

    TIter nextKey(file->GetListOfKeys());
    TKey *key;

    while ((key = (TKey*)nextKey()))
    {
        // Get the name and class of the object
        const char* name = key->GetName();
        const char* className = key->GetClassName();

        // Check if the object is a TTree
        if (strcmp(className, "TTree") == 0)
        {
            std::cout << "TTree name: " << name << std::endl;
        }
    }

    std::cout << "-------------------------------------" << std::endl;

    TObjArray* branches = tree->GetListOfBranches();
    for (int i = 0; i < branches->GetEntries(); i++) {
        TBranch* branch = (TBranch*)branches->At(i);
        std::cout << "Branch name: " << branch->GetName() << std::endl;
    }

    std::cout << "-------------------------------------" << std::endl;
    
}

In [6]:
// read_file("ROOT_Results/Moller_like/Simulation_proton_and_proton.root")

# Import Packages

In [7]:
#include <ROOT/RDataFrame.hxx>
#include <TFile.h>
#include <TH1F.h>
#include <TTree.h>
#include <iostream>

In [8]:
void read_histogram(const std::string& filename)
{
    TFile *file = TFile::Open(filename.c_str());

    std::cout << "reading the file:" << filename.c_str() << std::endl;

    if (!file || file->IsZombie())
    {
        std::cerr << "Failed to open ROOT file" << std::endl;
        return;
    }

    std::cout << "successfully opened the file" << std::endl;

    TTree *tree = (TTree*)file->Get("tree");

    if (!tree)
    {
        std::cerr << "Tree not found!" << std::endl;
        return;
    }

    std::cout << "successfully opened the tree" << std::endl;

    std::cout << "Tree has " << tree->GetEntries() << " entries." << std::endl;

    int id, event, size, no;
    double m, px, py, pz;
    tree->SetBranchAddress("size", &size);
    tree->SetBranchAddress("m", &m);
    tree->SetBranchAddress("px", &px);
    tree->SetBranchAddress("py", &py);
    tree->SetBranchAddress("pz", &pz);

    TH1F *hist_mom_x = new TH1F("X Momentum Histogram", "Histogram of momentum in x", 100, -10, 10);
    TH1F *hist_mom_y = new TH1F("Y Momentum Histogram", "Histogram of momentum in y", 100, -10, 10);
    TH1F *hist_mom_z = new TH1F("Z Momentum Histogram", "Histogram of momentum in z", 100, -100, 100);
    TH1F *hist_total_mom = new TH1F("Total Momentum Histogram", "Histogram of total momentum", 10, -10, 20);
    TH1F *hist_theta = new TH1F("Theta Histogram", "Histogram of scattering angle (theta)", 100, -TMath::Pi(), TMath::Pi());
    TH1F *hist_dsigma = new TH1F("Differential Cross Section", "Differential Cross Section dσ/dΩ", 100, -TMath::Pi(), TMath::Pi());
    TH1F *hist_total_cross_section = new TH1F("Total Cross Section", "Cumulative Total Cross Section", 100, 0, TMath::Pi());
    
    Long64_t nEntries = tree->GetEntries();
    for (Long64_t i = 0; i < nEntries; ++i)
    {
        tree->GetEntry(i);

        hist_mom_x->Fill(px);
        hist_mom_y->Fill(py);
        hist_mom_z->Fill(pz);
        double pabs = sqrt(pow(px, 2) + pow(py, 2) + pow(pz, 2));
        hist_total_mom->Fill(pabs);
        double theta = acos(pz / pabs);
        hist_theta->Fill(theta);
    }

    std::cout << "successfully filled the histogram with the tree values" << std::endl;

    double cumulative_cross_section = 0.0;

    for (int i = 1; i <= hist_theta->GetNbinsX(); ++i)
    {
        double theta = hist_theta->GetBinCenter(i);
        double bin_width = hist_theta->GetBinWidth(i);
    
        // Calculate solid angle for each bin in theta
        double solid_angle = 2 * TMath::Pi() * sin(theta) * bin_width;
    
        // Get the number of counts in the bin
        double count = hist_theta->GetBinContent(i);
    
        // Assume flux normalization (1 for simplicity, adjust as needed)
        double flux = 1.0;
    
        // Calculate the differential cross section (you can scale by the flux here)
        double dsigma = count / (flux * solid_angle);
    
        // Fill the differential cross section histogram
        hist_dsigma->SetBinContent(i, dsigma);

        // Update the cumulative cross section
        cumulative_cross_section += dsigma * solid_angle;
    
        // Fill the cumulative cross section histogram
        hist_total_cross_section->SetBinContent(i, cumulative_cross_section);
    }

    std::cout << "successfully computed the cross sections" << std::endl;

    TCanvas *canv = new TCanvas(filename.c_str(), "Canvas Title", 800, 600);

    if(hist_total_mom)
    {
        // hist_mom_x->Draw();
        // hist_mom_y->Draw();
        // hist_mom_z->Draw();
        // hist_total_mom->Draw();
        // hist_theta->Draw();
        // hist_dsigma->Draw();
        hist_total_cross_section->Draw();

        canv->Draw();
    }
    else
    {
        std::cerr << "Histogram not found in the tree!" << std::endl;
    }

    // file->Close();
}

# Moller like

$p^0$

In [9]:
read_histogram("ROOT_Results/Moller_like/Simulation_proton_and_proton.root");

reading the file:ROOT_Results/Moller_like/Simulation_proton_and_proton.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^+$

In [10]:
read_histogram("ROOT_Results/Moller_like/Simulation_sigma_plus_and_sigma_plus.root");

reading the file:ROOT_Results/Moller_like/Simulation_sigma_plus_and_sigma_plus.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^{++}_c$

In [11]:
read_histogram("ROOT_Results/Moller_like/Simulation_sigma_plus_plus_c_and_sigma_plus_plus_c.root");

reading the file:ROOT_Results/Moller_like/Simulation_sigma_plus_plus_c_and_sigma_plus_plus_c.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^{*+}_b$

In [12]:
read_histogram("ROOT_Results/Moller_like/Simulation_sigma_asterisk_plus_b_and_sigma_asterisk_plus_b.root");

reading the file:ROOT_Results/Moller_like/Simulation_sigma_asterisk_plus_b_and_sigma_asterisk_plus_b.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$n^0$

In [13]:
read_histogram("ROOT_Results/Moller_like/Simulation_neutron_and_neutron.root");

reading the file:ROOT_Results/Moller_like/Simulation_neutron_and_neutron.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^-$

In [14]:
read_histogram("ROOT_Results/Moller_like/Simulation_sigma_minus_and_sigma_minus.root");

reading the file:ROOT_Results/Moller_like/Simulation_sigma_minus_and_sigma_minus.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Xi^{0}_c$

In [15]:
read_histogram("ROOT_Results/Moller_like/Simulation_xi_zero_c_and_xi_zero_c.root");

reading the file:ROOT_Results/Moller_like/Simulation_xi_zero_c_and_xi_zero_c.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^{-}_b$

In [16]:
read_histogram("ROOT_Results/Moller_like/Simulation_sigma_minus_b_and_sigma_minus_b.root");

reading the file:ROOT_Results/Moller_like/Simulation_sigma_minus_b_and_sigma_minus_b.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Xi^0$

In [17]:
read_histogram("ROOT_Results/Moller_like/Simulation_xi_zero_and_xi_zero.root");

reading the file:ROOT_Results/Moller_like/Simulation_xi_zero_and_xi_zero.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Xi^-$

In [18]:
read_histogram("ROOT_Results/Moller_like/Simulation_xi_minus_and_xi_minus.root");

reading the file:ROOT_Results/Moller_like/Simulation_xi_minus_and_xi_minus.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Omega^{0}_c$

In [19]:
read_histogram("ROOT_Results/Moller_like/Simulation_omega_0_c_and_omega_0_c.root");

reading the file:ROOT_Results/Moller_like/Simulation_omega_0_c_and_omega_0_c.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Omega^{-}_b$

In [20]:
read_histogram("ROOT_Results/Moller_like/Simulation_omega_minus_b_and_omega_minus_b.root");

reading the file:ROOT_Results/Moller_like/Simulation_omega_minus_b_and_omega_minus_b.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


# Compton like

$p^+ + \pi^0$

In [21]:
read_histogram("ROOT_Results/Compton_like/Simulation_proton_and_pion_0.root");

reading the file:ROOT_Results/Compton_like/Simulation_proton_and_pion_0.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$p^+ + \pi^+$

In [22]:
read_histogram("ROOT_Results/Compton_like/Simulation_proton_and_pion_+.root");

reading the file:ROOT_Results/Compton_like/Simulation_proton_and_pion_+.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$p^+ + \pi^-$

In [23]:
read_histogram("ROOT_Results/Compton_like/Simulation_proton_and_pion_-.root");

reading the file:ROOT_Results/Compton_like/Simulation_proton_and_pion_-.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^+ + \pi^0$

In [24]:
read_histogram("ROOT_Results/Compton_like/Simulation_sigma_plus_and_pion_0.root");

reading the file:ROOT_Results/Compton_like/Simulation_sigma_plus_and_pion_0.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^+ + K^+$

In [25]:
read_histogram("ROOT_Results/Compton_like/Simulation_sigma_plus_and_K_+.root");

reading the file:ROOT_Results/Compton_like/Simulation_sigma_plus_and_K_+.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^+ + K^-$

In [26]:
read_histogram("ROOT_Results/Compton_like/Simulation_sigma_plus_and_K_-.root");

reading the file:ROOT_Results/Compton_like/Simulation_sigma_plus_and_K_-.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^{++}_c + \pi^{0}$

In [27]:
read_histogram("ROOT_Results/Compton_like/Simulation_sigma_plus_plus_c_and_pion_0.root");

reading the file:ROOT_Results/Compton_like/Simulation_sigma_plus_plus_c_and_pion_0.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^{++}_c + D^{0}$

In [28]:
read_histogram("ROOT_Results/Compton_like/Simulation_sigma_plus_plus_c_and_D_0.root");

reading the file:ROOT_Results/Compton_like/Simulation_sigma_plus_plus_c_and_D_0.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^{++}_c + \bar{D}^{0}$

In [29]:
read_histogram("ROOT_Results/Compton_like/Simulation_sigma_plus_plus_c_and_D_bar_0.root");

reading the file:ROOT_Results/Compton_like/Simulation_sigma_plus_plus_c_and_D_bar_0.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^{*+}_b + \pi^0$

In [30]:
read_histogram("ROOT_Results/Compton_like/Simulation_sigma_asterisk_plus_b_and_pion_0.root");

reading the file:ROOT_Results/Compton_like/Simulation_sigma_asterisk_plus_b_and_pion_0.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^{*+}_b + B^+$

In [31]:
read_histogram("ROOT_Results/Compton_like/Simulation_sigma_asterisk_plus_b_and_B_+.root");

reading the file:ROOT_Results/Compton_like/Simulation_sigma_asterisk_plus_b_and_B_+.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^{*+}_b + B^-$

In [32]:
read_histogram("ROOT_Results/Compton_like/Simulation_sigma_asterisk_plus_b_and_B_-.root");

reading the file:ROOT_Results/Compton_like/Simulation_sigma_asterisk_plus_b_and_B_-.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$n^0 + \pi^-$

In [33]:
read_histogram("ROOT_Results/Compton_like/Simulation_neutron_and_pion_-.root");

reading the file:ROOT_Results/Compton_like/Simulation_neutron_and_pion_-.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$n^0 + \pi^+$

In [34]:
read_histogram("ROOT_Results/Compton_like/Simulation_neutron_and_pion_+.root");

reading the file:ROOT_Results/Compton_like/Simulation_neutron_and_pion_+.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^- + K^{0}$

In [35]:
read_histogram("ROOT_Results/Compton_like/Simulation_sigma_minus_and_K_0.root");

reading the file:ROOT_Results/Compton_like/Simulation_sigma_minus_and_K_0.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^- + \bar{K}^{0}$

In [36]:
read_histogram("ROOT_Results/Compton_like/Simulation_sigma_minus_and_K_bar_0.root");

reading the file:ROOT_Results/Compton_like/Simulation_sigma_minus_and_K_bar_0.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Xi^{0}_c + D^-$

In [37]:
read_histogram("ROOT_Results/Compton_like/Simulation_xi_zero_c_and_D_-.root");

reading the file:ROOT_Results/Compton_like/Simulation_xi_zero_c_and_D_-.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Xi^{0}_c + D^+$

In [38]:
read_histogram("ROOT_Results/Compton_like/Simulation_xi_zero_c_and_D_+.root");

reading the file:ROOT_Results/Compton_like/Simulation_xi_zero_c_and_D_+.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^{-}_{b} + B^{0}$

In [39]:
read_histogram("ROOT_Results/Compton_like/Simulation_sigma_minus_b_and_B_0.root");

reading the file:ROOT_Results/Compton_like/Simulation_sigma_minus_b_and_B_0.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Sigma^{-}_{b} + \bar{B}^{0}$

In [40]:
read_histogram("ROOT_Results/Compton_like/Simulation_sigma_minus_b_and_B_bar_0.root");

reading the file:ROOT_Results/Compton_like/Simulation_sigma_minus_b_and_B_bar_0.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Xi^0 + \eta$

In [41]:
read_histogram("ROOT_Results/Compton_like/Simulation_xi_zero_and_eta.root");

reading the file:ROOT_Results/Compton_like/Simulation_xi_zero_and_eta.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Xi^0 + K^-$

In [42]:
read_histogram("ROOT_Results/Compton_like/Simulation_xi_zero_and_K_-.root");

reading the file:ROOT_Results/Compton_like/Simulation_xi_zero_and_K_-.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Xi^0 + K^+$

In [43]:
read_histogram("ROOT_Results/Compton_like/Simulation_xi_zero_and_K_+.root");

reading the file:ROOT_Results/Compton_like/Simulation_xi_zero_and_K_+.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Xi^- + \eta$

In [44]:
read_histogram("ROOT_Results/Compton_like/Simulation_xi_minus_and_eta.root");

reading the file:ROOT_Results/Compton_like/Simulation_xi_minus_and_eta.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Xi^- + K^0$

In [45]:
read_histogram("ROOT_Results/Compton_like/Simulation_xi_minus_and_K_0.root");

reading the file:ROOT_Results/Compton_like/Simulation_xi_minus_and_K_0.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Xi^- + \bar{K}^0$

In [46]:
read_histogram("ROOT_Results/Compton_like/Simulation_xi_minus_and_K_bar_0.root");

reading the file:ROOT_Results/Compton_like/Simulation_xi_minus_and_K_bar_0.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Omega^{0}_c + \eta$

In [47]:
read_histogram("ROOT_Results/Compton_like/Simulation_omega_0_c_and_eta.root");

reading the file:ROOT_Results/Compton_like/Simulation_omega_0_c_and_eta.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Omega^{0}_c + D^{-}_s$

In [48]:
read_histogram("ROOT_Results/Compton_like/Simulation_omega_0_c_and_D_-_s.root");

reading the file:ROOT_Results/Compton_like/Simulation_omega_0_c_and_D_-_s.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Omega^{0}_c + D^{+}_s$

In [49]:
read_histogram("ROOT_Results/Compton_like/Simulation_omega_0_c_and_D_+_s.root");

reading the file:ROOT_Results/Compton_like/Simulation_omega_0_c_and_D_+_s.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Omega^{-}_b + \eta$

In [50]:
read_histogram("ROOT_Results/Compton_like/Simulation_omega_minus_b_and_eta.root");

reading the file:ROOT_Results/Compton_like/Simulation_omega_minus_b_and_eta.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Omega^{-}_b + B^{0}_s$

In [51]:
read_histogram("ROOT_Results/Compton_like/Simulation_omega_minus_b_and_B_0_s.root");

reading the file:ROOT_Results/Compton_like/Simulation_omega_minus_b_and_B_0_s.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections


$\Omega^{-}_b + \bar{B}^{0}_s$

In [52]:
read_histogram("ROOT_Results/Compton_like/Simulation_omega_minus_b_and_B_bar_0_s.root");

reading the file:ROOT_Results/Compton_like/Simulation_omega_minus_b_and_B_bar_0_s.root
successfully opened the file
successfully opened the tree
Tree has 10000 entries.
successfully filled the histogram with the tree values
successfully computed the cross sections
